In [1]:
import pandas as pd
from pyaxis import pyaxis

px = pyaxis.parse(uri= "daten.px", encoding = "utf-8")
df = px["DATA"]
meta= px["METADATA"]

Multilingual PX file


In [2]:
df.columns


Index(['Straftat', 'Kanton', 'Aufenthaltsgruppe', 'Geschlecht', 'Altersklasse',
       'Jahr', 'DATA'],
      dtype='str')

In [3]:
for i in df["Straftat"].unique(): 
    if "dieb" in i.lower():
        print(i)
for i in df["Altersklasse"].unique(): 
    print(i)
for g in df["Aufenthaltsgruppe"].unique():
    print(g)
for k in df["Geschlecht"].unique():
    print(k)
for p in df["Jahr"].unique():
    print(p)
for w in df["Kanton"].unique():
    print(w)


Diebstahl (Art. 139)
Einbruchdiebstahl (Art. 139)
Einschleichdiebstahl (Art. 139)
Entreissdiebstahl (Art. 139)
Ladendiebstahl (Art. 139)
Taschendiebstahl (Art. 139)
Trickdiebstahl (Art. 139)
Fahrzeugeinbruchdiebstahl (Art. 139)
Fahrzeugdiebstahl (Art. 139)
Diebstahl ab/aus Fahrzeug (Art. 139)
Sachbeschädigung + Diebstahl (Art. 144)
Hausfriedensbruch + Diebstahl (Art. 186)
Altersklasse - Total
<10 Jahre
10-14 Jahre
15-17 Jahre
18-19 Jahre
20-24 Jahre
25-29 Jahre
30-34 Jahre
35-39 Jahre
40-44 Jahre
45-49 Jahre
50-54 Jahre
55-59 Jahre
60-64 Jahre
65-69 Jahre
70-79 Jahre
80-89 Jahre
90 Jahre und +
ohne Angabe
Total alle Beschuldigte
Total Schweizer Beschuldigte
Total Beschuldigte aus der ständigen ausländischen Wohnbevölkerung
Total Beschuldigte aus der Asylbevölkerung
Total übrige ausländische Beschuldigte
Geschlecht - Total
Mann
Frau
2009
2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021
2022
2023
2024
2025
Schweiz
Zürich
Bern / Berne
Luzern
Uri
Schwyz
Obwalden
Nidwalden
Glarus

In [4]:
# =============================================================
# Gemeinsame Vorbereitung für alle drei Datensätze
#   - Straftat: Ladendiebstahl (Art. 139)
#   - Kanton: alle 26 Kantone (ohne "Schweiz")
#   - Aufenthaltsgruppe: Schweizer + ständige ausländische Wohnbevölkerung
#   - Geschlecht: Total
#   - Jahr: alle
# =============================================================
import pandas as pd
import numpy as np
from pyaxis import pyaxis

# Daten einlesen
px = pyaxis.parse(uri="Daten.px", encoding="utf-8")
df_raw = px["DATA"]          # Originaldaten – werden NIE verändert
meta = px["METADATA"]

# Kopie erstellen
df = df_raw.copy()

# Gemeinsame Filter
ständig = [
    "Total Schweizer Beschuldigte",
    "Total Beschuldigte aus der ständigen ausländischen Wohnbevölkerung",
]

df_basis = df.loc[
    (df["Straftat"] == "Ladendiebstahl (Art. 139)") &   # nur Ladendiebstahl
    (df["Kanton"] != "Schweiz") &                        # alle 26 Kantone
    (df["Aufenthaltsgruppe"].isin(ständig)) &            # ständige Wohnbevölkerung
    (df["Geschlecht"] == "Geschlecht - Total")           # beide Geschlechter zusammen
].copy()

# "......" (Datenschutz) und andere Platzhalter werden zu NaN
df_basis["DATA"] = pd.to_numeric(df_basis["DATA"], errors="coerce")
df_basis["Jahr"] = pd.to_numeric(df_basis["Jahr"])

# " Jahre" aus den Altersklassen entfernen: "20-24 Jahre" -> "20-24"
df_basis["Altersklasse"] = df_basis["Altersklasse"].str.replace(" Jahre", "", regex=False)

# Kantonsnamen auf Deutsch
kanton_deutsch = {
    "Bern / Berne": "Bern",
    "Fribourg / Freiburg": "Freiburg",
    "Graubünden / Grigioni / Grischun": "Graubünden",
    "Ticino": "Tessin",
    "Vaud": "Waadt",
    "Valais / Wallis": "Wallis",
    "Neuchâtel": "Neuenburg",
    "Genève": "Genf",
}
df_basis["Kanton"] = df_basis["Kanton"].replace(kanton_deutsch)

print("df_basis:", df_basis.shape)
print(sorted(df_basis["Kanton"].unique()))
print("Anzahl Kantone:", df_basis["Kanton"].nunique())   # sollte 26 sein

Multilingual PX file
df_basis: (16796, 7)
['Aargau', 'Appenzell Ausserrhoden', 'Appenzell Innerrhoden', 'Basel-Landschaft', 'Basel-Stadt', 'Bern', 'Freiburg', 'Genf', 'Glarus', 'Graubünden', 'Jura', 'Luzern', 'Neuenburg', 'Nidwalden', 'Obwalden', 'Schaffhausen', 'Schwyz', 'Solothurn', 'St. Gallen', 'Tessin', 'Thurgau', 'Uri', 'Waadt', 'Wallis', 'Zug', 'Zürich']
Anzahl Kantone: 26


In [5]:
# =============================================================
# Datensatz A: Altersklassen 20-24 und 25-29 GETRENNT
#   eine Zeile pro Kanton × Jahr × Altersklasse
#   Summe über die 2 Aufenthaltsgruppen
# =============================================================
df_A = (
    df_basis.loc[df_basis["Altersklasse"].isin(["20-24", "25-29"])]
    .groupby(["Kanton", "Jahr", "Altersklasse"], dropna=False)["DATA"]
    .sum(min_count=2)                  # fehlt eine der 2 Gruppen -> NaN
    .reset_index()
    .rename(columns={"DATA": "Beschuldigte"})
)

# Kontrolle
print("Zeilen:", len(df_A), "(erwartet: 884)")
print("Fehlende Werte:", df_A["Beschuldigte"].isna().sum(),
      f"({df_A['Beschuldigte'].isna().mean():.0%})")
fehlt_A = df_A.groupby("Kanton")["Beschuldigte"].apply(lambda s: s.isna().mean()).round(2)
print("Kantone mit Lücken:")
print(fehlt_A[fehlt_A > 0].sort_values(ascending=False).to_string())

# Speichern
df_A.to_csv("df_A_20-24_25-29_getrennt.csv", index=False)
df_A.head()

Zeilen: 884 (erwartet: 884)
Fehlende Werte: 281 (32%)
Kantone mit Lücken:
Kanton
Appenzell Ausserrhoden    1.00
Glarus                    1.00
Nidwalden                 1.00
Obwalden                  1.00
Uri                       1.00
Jura                      0.97
Appenzell Innerrhoden     0.94
Schaffhausen              0.71
Tessin                    0.35
Schwyz                    0.24
Graubünden                0.03
Zug                       0.03


,Kanton,Jahr,Altersklasse,Beschuldigte
0,Aargau,2009,20-24,45.0
1,Aargau,2009,25-29,37.0
2,Aargau,2010,20-24,41.0
3,Aargau,2010,25-29,37.0
4,Aargau,2011,20-24,40.0


In [7]:
# =============================================================
# Datensatz B: Altersklassen 20-24 und 25-29 KOMBINIERT (20-29)
#   eine Zeile pro Kanton × Jahr
#   Summe über 2 Aufenthaltsgruppen × 2 Altersklassen = 4 Zellen
# =============================================================
df_B = (
    df_basis.loc[df_basis["Altersklasse"].isin(["20-24", "25-29"])]
    .groupby(["Kanton", "Jahr"], dropna=False)["DATA"]
    .sum(min_count=4)                  # fehlt eine der 4 Zellen -> NaN
    .reset_index()
    .rename(columns={"DATA": "Beschuldigte"})
)
df_B["Altersklasse"] = "20-29"
df_B = df_B[["Kanton", "Jahr", "Altersklasse", "Beschuldigte"]]   # gleiche Spaltenreihenfolge wie A

# Kontrolle
print("Zeilen:", len(df_B), "(erwartet: 442)")
print("Fehlende Werte:", df_B["Beschuldigte"].isna().sum(),
      f"({df_B['Beschuldigte'].isna().mean():.0%})")
fehlt_B = df_B.groupby("Kanton")["Beschuldigte"].apply(lambda s: s.isna().mean()).round(2)
print("Kantone mit Lücken:")
print(fehlt_B[fehlt_B > 0].sort_values(ascending=False).to_string())

# Speichern
df_B.to_csv("df_B_20-29_kombiniert.csv", index=False)
df_B.head()

Zeilen: 442 (erwartet: 442)
Fehlende Werte: 147 (33%)
Kantone mit Lücken:
Kanton
Appenzell Ausserrhoden    1.00
Glarus                    1.00
Jura                      1.00
Nidwalden                 1.00
Obwalden                  1.00
Uri                       1.00
Appenzell Innerrhoden     0.94
Schaffhausen              0.82
Tessin                    0.41
Schwyz                    0.35
Graubünden                0.06
Zug                       0.06


,Kanton,Jahr,Altersklasse,Beschuldigte
0,Aargau,2009,20-29,82.0
1,Aargau,2010,20-29,78.0
2,Aargau,2011,20-29,78.0
3,Aargau,2012,20-29,104.0
4,Aargau,2013,20-29,100.0


In [8]:
# =============================================================
# Datensatz C: ALLE Altersklassen ("Altersklasse - Total")
#   eine Zeile pro Kanton × Jahr
#   Summe über die 2 Aufenthaltsgruppen
# =============================================================
df_C = (
    df_basis.loc[df_basis["Altersklasse"] == "Altersklasse - Total"]
    .groupby(["Kanton", "Jahr"], dropna=False)["DATA"]
    .sum(min_count=2)                  # fehlt eine der 2 Gruppen -> NaN
    .reset_index()
    .rename(columns={"DATA": "Beschuldigte"})
)

# Kontrolle
print("Zeilen:", len(df_C), "(erwartet: 442)")
print("Fehlende Werte:", df_C["Beschuldigte"].isna().sum(),
      f"({df_C['Beschuldigte'].isna().mean():.0%})")
fehlt_C = df_C.groupby("Kanton")["Beschuldigte"].apply(lambda s: s.isna().mean()).round(2)
print("Kantone mit Lücken:")
print(fehlt_C[fehlt_C > 0].sort_values(ascending=False).to_string())

# Speichern
df_C.to_csv("df_C_Alter_total.csv", index=False)
df_C.head()

Zeilen: 442 (erwartet: 442)
Fehlende Werte: 23 (5%)
Kantone mit Lücken:
Kanton
Appenzell Innerrhoden     0.82
Uri                       0.29
Obwalden                  0.12
Appenzell Ausserrhoden    0.06
Nidwalden                 0.06


,Kanton,Jahr,Beschuldigte
0,Aargau,2009,495.0
1,Aargau,2010,562.0
2,Aargau,2011,524.0
3,Aargau,2012,551.0
4,Aargau,2013,470.0


In [9]:
print(df_A.head())
print(df_B.head())
print(df_C.head())

   Kanton  Jahr Altersklasse  Beschuldigte
0  Aargau  2009        20-24          45.0
1  Aargau  2009        25-29          37.0
2  Aargau  2010        20-24          41.0
3  Aargau  2010        25-29          37.0
4  Aargau  2011        20-24          40.0
   Kanton  Jahr Altersklasse  Beschuldigte
0  Aargau  2009        20-29          82.0
1  Aargau  2010        20-29          78.0
2  Aargau  2011        20-29          78.0
3  Aargau  2012        20-29         104.0
4  Aargau  2013        20-29         100.0
   Kanton  Jahr  Beschuldigte
0  Aargau  2009         495.0
1  Aargau  2010         562.0
2  Aargau  2011         524.0
3  Aargau  2012         551.0
4  Aargau  2013         470.0
